# Intro to Machine Learning
What is machine learning? "Machine learning is the science of getting computers to act without being explicitly programmed." ~Andrew Ng

Two main types (to start with):
1. Supervised learning: working with *labeled data*
    * Labeled data: data that has an attribute (column, feature) you want to predict for new/unseen instances (rows, samples)
        * The attribute is called the "class"
        * If the class attribute is categorical, then the supervised learning task is called *classification* (a category into which something is put.)
        * If the class attribute is numeric, then the supervised learning task is called *regression* (a measure of the relation between the mean value of one variable (e.g. output) and corresponding values of other variables (e.g. time and cost).)
    * **We are going to learn the basics of classification**
1. Unsupervised learning: working with *unlabeled data*
    * Unlabeled data: there is no such class attribute (you are interested in predicting)

## Supervised Learning w/k Nearest Neighbors Algorithm
* Suppose we have the height, weight, and T-shirt size for 4 people
    * Height and weight are our attributes
    * T-shirt size is our class we want to predict given a new 5th person's height and weight
    * The 4 people we have height, weight, and T-shirt size information for form our "training set"
        * The training set is used to build/train a supervised machine learning algorithm/model
            * *Like when your teacher gives you examples for you to learn from*
    * The new 5th person is an "unseen instance". 1 or more unseen instances form a "testing set"
        * The testing set is used to determine how good the algorithm/model is (how much did the algorithm/model learn from the training set?)
            * *Like when your teacher gives you unseen examples (training set) on an exam to see how much you have learned*
            * *If you do well on the exam, the teacher thinks you have learned from the training set*
            * *If you do not do well on the exam, the teacher thinks you haven't learned from the training set, and will probably give you new examples to learn from and a new exam 😀*
    * Data science is used to create the training and testing sets, and visualize/understand the testing results

## k Nearest Neighbors
* Identify the k nearest neighbors of an unseen instance to each instance in the training set 
    * The most frequent class label amongst the k closest neighbors is the prediction for this unseen instance's class label
* We need a way to measure "nearness" AKA "closeness"
    * 2D: Pythagorean theorem
    * ND: Euclidean distance $dist(a, b) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$
* We need to normalize (AKA scale) our attributes so that we don't have an inadvertent weighting of one attribute more than the others
    * Example: height is on a larger scale than weight, it could dominate the formula
    * We will use min-max scaling
        * For each attribute, from each value we substract the min and divide by the original range (max - min)
        * For each attribute, the min goes to 0 and the max goes to 1 (bounds the attribute to [0, 1])

# kNN Coding Example
## Load the Data

In [7]:
import pandas as pd

df = pd.read_csv("shirt_sizes.csv")
print(df)

   height(cm)  weight(kg) t-shirt size
0         158          58            M
1         163          61            M
2         165          61            L
3         168          66            L


## Prepare the Training Set
For use with the Sci-kit Learn library: https://scikit-learn.org/stable/getting_started.html
* X: a matrix (2 dimensional data structure, like a table, a 2D list, or a DataFrame)
    * Stores our instances/features (but not the class attribute)
* y: a vector (1 dimensional data structure, like a single column, a 1D list, or a Series)
    * Stores our class attribute (what we want to predict)

In [8]:
X_train = df.drop(["t-shirt size"], axis=1) # 1 means drop column
print(X_train)
y_train = df["t-shirt size"]
print(y_train)

   height(cm)  weight(kg)
0         158          58
1         163          61
2         165          61
3         168          66
0    M
1    M
2    L
3    L
Name: t-shirt size, dtype: object


## Normalize the Data
By applying min-max scaling approach

Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html

In [9]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(X_train) # so the scaler can figure out the min, max, range
X_train_scaled = scaler.transform(X_train) # so the scaler can scale X_train
print(X_train_scaled)

# set up unseen instance in test set
X_test = [
    [161, 63]
]
X_test_scaled = scaler.transform(X_test)
print(X_test_scaled)

[[0.    0.   ]
 [0.5   0.375]
 [0.7   0.375]
 [1.    1.   ]]
[[0.3   0.625]]


## Make a Prediction!!
Using X_train_scaled and X_test_scaled

Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

In [10]:
from sklearn.neighbors import KNeighborsClassifier

# train the kNN algorithm using fit() method
clf = KNeighborsClassifier(n_neighbors=3, metric="euclidean") # euclidean for pythagorean theorem
clf.fit(X_train_scaled, y_train)

# predict using the kNN algorithm using predict() method
y_predicted = clf.predict(X_test_scaled)
print(y_predicted)

['M']


## Again, but with a Larger Dataset
From shirt_sizes_long.csv
* So our testing set has more than one unseen instance in it

In [36]:
df = pd.read_csv("shirt_sizes_long.csv")
print(df)
X = df.drop("t-shirt size", axis=1)
y = df["t-shirt size"] # our class attribute we want to predict
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
print(X)

    height(cm)  weight(kg) t-shirt size
0          158          58            M
1          158          59            M
2          158          63            M
3          160          59            M
4          160          60            M
5          163          60            M
6          163          61            M
7          160          64            L
8          163          64            L
9          165          61            L
10         165          62            L
11         165          65            L
12         168          62            L
13         168          63            L
14         168          66            L
15         170          63            L
16         170          64            L
17         170          68            L
[[0.         0.        ]
 [0.         0.1       ]
 [0.         0.5       ]
 [0.16666667 0.1       ]
 [0.16666667 0.2       ]
 [0.41666667 0.2       ]
 [0.41666667 0.3       ]
 [0.16666667 0.6       ]
 [0.41666667 0.6       ]
 [0.58333333 0.

## Divide Dataset using "Train/Test Split"
Also known as the "hold out method"
* "Hold out" (or remove) some instances for testing
* We train on the remaining instances (that were not held out for testing)
* The question to answer then is: *how many instances to hold out (or reserve) for testing?*
    * Typically do a 2:1 train:test split
        * Hold out 1/3 (33%) of the data for testing
        * Train on the remaining 2/3 (67%) of the data
        * Example: 18 instances in shirt_sizes_long.csv, then 1/3 * 18 -> 6 instances for testing
    * Typically hold out 25% for testing (remaining 75% for training)
        * This is the default for Sci-kit Learn's train_test_split() function that implements this hold out method
        * Example 0.25 * 18 -> 5 instances for testing

Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, 
    test_size=0.25, # 25% hold out
    random_state=0, # like seeding the random number generator
    stratify=y) # make sure the training/testing sets have a similar
    # distribution (or spread) of Ms and Ls as the original dataset 

print("held out for testing:")
# print(X_test)
# print(y_test)

# get predictions for each instance in the test set
clf.fit(X_train, y_train)
y_predicted = clf.predict(X_test)
print("y_predicted:\t", list(y_predicted))
print("y_test:\t\t", y_test.tolist())

# the kNN algorithm predicted 4/5 testing set instance's t-shirt sizes
# correctly... therefore the accuracy is 80%
# accuracy is a way to measure how good a classifier does on a testing set
# accuracy = number correctly predicted / total number of predictions made
# using sci-kit learn library
accuracy = clf.score(X_test, y_test)
print("accuracy:", accuracy)

held out for testing:
y_predicted:	 ['L', 'M', 'M', 'M', 'L']
y_test:		 ['L', 'M', 'L', 'M', 'L']
accuracy: 0.8
